In [0]:
print("olá, Databricks")

olá, Databricks


In [0]:
spark.range(5).show()

+---+
| id|
+---+
|  0|
|  1|
|  2|
|  3|
|  4|
+---+



In [0]:
spark.sql("SELECT current_catalog(), current_schema()").show()

+-----------------+----------------+
|current_catalog()|current_schema()|
+-----------------+----------------+
|        workspace|         default|
+-----------------+----------------+



Bloco 3: do CSV à tabela Delta

Analogia: o catálogo workspace é o prédio. O schema é uma gaveta do projeto. O volume é uma pasta de arquivos soltos dentro da gaveta, onde o CSV fica como veio. A tabela Delta é a planilha organizada feita a partir dele.



3.1 Criar a gaveta.

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.process_sigma")

DataFrame[]

3.2 Criar a pasta de arquivos:

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.process_sigma.raw")

DataFrame[]

3.4 Ler o arquivo e olhar antes de gravar:

In [0]:
df = spark.read.csv("/Volumes/workspace/process_sigma/raw/template_injecao.csv", header=True)
df.printSchema()
df.show(5, truncate=False)
print(df.count())

root
 |-- Order Id: string (nullable = true)
 |-- order date (DateOrders): string (nullable = true)
 |-- shipping date (DateOrders): string (nullable = true)
 |-- Days for shipment (scheduled): string (nullable = true)
 |-- Days for shipping (real): string (nullable = true)
 |-- Delivery Status: string (nullable = true)
 |-- Late_delivery_risk: string (nullable = true)
 |-- Order Region: string (nullable = true)
 |-- Order Country: string (nullable = true)
 |-- Category Name: string (nullable = true)

+--------+-----------------------+--------------------------+-----------------------------+------------------------+----------------+------------------+-------------+-------------+-------------+
|Order Id|order date (DateOrders)|shipping date (DateOrders)|Days for shipment (scheduled)|Days for shipping (real)|Delivery Status |Late_delivery_risk|Order Region |Order Country|Category Name|
+--------+-----------------------+--------------------------+-----------------------------+------------

3.5 Gravar como tabela Delta:

In [0]:
from pyspark.sql import functions as F

COLUMNS_MAP = {
    "Order Id": "order_id",
    "order date (DateOrders)": "order_date",
    "shipping date (DateOrders)": "ship_date",
    "Days for shipment (scheduled)": "days_scheduled",
    "Days for shipping (real)": "days_real",
    "Delivery Status": "delivery_status",
    "Late_delivery_risk": "late_delivery_risk",
    "Order Region": "order_region",
    "Order Country": "order_country",
    "Category Name": "category_name",
}

bronze = df
for old, new in COLUMNS_MAP.items():
    bronze = bronze.withColumnRenamed(old, new)

bronze = (
    bronze
    .withColumn("source", F.lit("csv_upload"))
    .withColumn("ingested_at", F.current_timestamp())
)

bronze.write.mode("overwrite").saveAsTable("workspace.process_sigma.bronze_orders")

3.6 Conferir

In [0]:
spark.table("workspace.process_sigma.bronze_orders").printSchema()
print(spark.table("workspace.process_sigma.bronze_orders").count())
spark.sql("DESCRIBE HISTORY workspace.process_sigma.bronze_orders").show(truncate=False)

root
 |-- order_id: string (nullable = true)
 |-- order_date: string (nullable = true)
 |-- ship_date: string (nullable = true)
 |-- days_scheduled: string (nullable = true)
 |-- days_real: string (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_delivery_risk: string (nullable = true)
 |-- order_region: string (nullable = true)
 |-- order_country: string (nullable = true)
 |-- category_name: string (nullable = true)
 |-- source: string (nullable = true)
 |-- ingested_at: timestamp (nullable = true)

500
+-------+-------------------+--------------+--------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------